In [1]:
from pathlib import Path
import re

import pandas as pd


# ============================================================
# Configuration
# ============================================================

DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed/notebooks")

WINDOW_SIZE = 60


# ============================================================
# Selected process variables
# ============================================================

SELECTED_VARIABLES = [
    # Temperature
    "T701", "T702", "T703", "T704", "T705",
    "T706", "T708", "T709", "T711", "T712",

    # Pressure
    "P701", "P702", "PDI701", "PDI702", "PY23",

    # Flow
    "FT703", "FT704", "FYI702",

    # Level
    "LS701", "LS702",

    # Heating / power
    "H002", "H701", "H702", "H704", "H706", "H708",

    # Vacuum
    "P301", "TV1",

    # Nitrogen
    "AV709",

    # Cooling
    "AV716",
]


# ============================================================
# Known scaling corrections
# ============================================================

SCALED_1000_COLUMNS = [
    "T701", "T702", "T703", "T704", "T706",
    "T708", "T709", "T711", "T712", "T705",
    "PY23", "TV1", "H702", "H706", "H708",
    "H704", "P701", "P702",
]

SCALED_1000_THRESHOLD = 500

SCALED_1000_SMALL_COLUMNS = [
    "FT703", "FT704", "PDI701", "PDI702",
]

SCALED_1000_SMALL_THRESHOLD = 1


# ============================================================
# Known timestamp error
# ============================================================

KNOWN_TIMESTAMP_ERROR = (
    "Shutdown/"
    "batch_dist_ternary_butan-1-ol+propan-2-ol+water/"
    "operating_point_013/"
    "train_normal/"
    "experiment_001"
)


# ============================================================
# Batch pattern
# ============================================================

BATCH_PATTERN = (
    r"(batch_dist_binary_ethanol\+propan-2-ol|"
    r"batch_dist_ternary_acetone\+butan-1-ol\+methanol|"
    r"batch_dist_ternary_butan-1-ol\+propan-2-ol\+water)"
)


# ============================================================
# Find file metadata
# ============================================================

def get_file_metadata(folder_name):

    records = []

    folder = DATA_DIR / folder_name

    for csv_path in folder.rglob("*.csv"):

        path_str = str(csv_path)

        phase_match = re.search(
            r"(Startup|Operation|Shutdown)",
            path_str,
            re.IGNORECASE,
        )

        batch_match = re.search(
            BATCH_PATTERN,
            path_str,
        )

        operating_point_match = re.search(
            r"operating_point_(\d+)",
            path_str,
        )

        experiment_match = re.search(
            r"(train_normal|test_anormal)_experiment_(\d+)",
            csv_path.stem,
        )

        if not (
            phase_match
            and batch_match
            and operating_point_match
            and experiment_match
        ):
            continue

        records.append({
            "file_path": str(csv_path),
            "phase": phase_match.group(1),
            "batch": batch_match.group(1),
            "operating_point": operating_point_match.group(0),
            "experiment_type": experiment_match.group(1),
            "experiment_number": (
                f"experiment_{experiment_match.group(2)}"
            ),
        })

    return pd.DataFrame(records)


# ============================================================
# Load time series
# ============================================================

def load_timeseries(folder_name):

    files = get_file_metadata(folder_name)

    data = []

    for _, row in files.iterrows():

        df = pd.read_csv(row["file_path"])

        df["identifier"] = "/".join([
            row["phase"],
            row["batch"],
            row["operating_point"],
            row["experiment_type"],
            row["experiment_number"],
        ])

        data.append(df)

    if not data:
        raise ValueError(
            f"No CSV files found in {folder_name}"
        )

    return pd.concat(
        data,
        ignore_index=True,
    )


# ============================================================
# Load raw data
# ============================================================

anomalies_ts = load_timeseries(
    "00_Timeseries_Label_Anomaly_Metadata"
)

sensors_ts = load_timeseries(
    "01_Timeseries_Sensors"
)

actuators_ts = load_timeseries(
    "02_Timeseries_Actuators"
)


# ============================================================
# Prepare anomaly labels
# ============================================================

anomalies_ts.loc[
    anomalies_ts["Time"].isna(),
    "Label (anomaly)"
] = 0

anomalies_ts = anomalies_ts.rename(
    columns={
        "Label (anomaly)": "anomaly_label"
    }
)


# ============================================================
# Merge data
# ============================================================

experiments = (
    actuators_ts
    .merge(
        anomalies_ts[
            ["identifier", "Time", "anomaly_label"]
        ],
        on=["identifier", "Time"],
        how="left",
    )
    .merge(
        sensors_ts,
        on=["identifier", "Time"],
        how="left",
    )
)

experiments["anomaly_label"] = (
    experiments["anomaly_label"]
    .fillna(0)
)


# ============================================================
# Create elapsed time
# ============================================================

experiments["Time_td"] = pd.to_timedelta(
    experiments["Time"],
    errors="coerce",
)

experiments["time_diff"] = (
    experiments
    .groupby("identifier")["Time_td"]
    .diff()
)

experiments["time_diff_clean"] = (
    experiments["time_diff"]
    .where(
        ~(
            (experiments["identifier"] == KNOWN_TIMESTAMP_ERROR)
            & (experiments["time_diff"] < pd.Timedelta(0))
        ),
        pd.Timedelta(0),
    )
    .fillna(pd.Timedelta(0))
)

experiments["elapsed_time"] = (
    experiments
    .groupby("identifier")["time_diff_clean"]
    .cumsum()
    .dt.total_seconds()
)

experiments = experiments.drop(
    columns=[
        "Time_td",
        "time_diff",
        "time_diff_clean",
    ]
)


# ============================================================
# Add metadata from identifier
# ============================================================

identifier_parts = experiments["identifier"].str.split(
    "/",
    expand=True,
)

identifier_parts.columns = [
    "phase",
    "batch",
    "operating_point",
    "experiment_type",
    "experiment",
]

experiments = pd.concat(
    [
        experiments,
        identifier_parts,
    ],
    axis=1,
)


# ============================================================
# Correct known scaling errors
# ============================================================

for columns, threshold in [
    (
        SCALED_1000_COLUMNS,
        SCALED_1000_THRESHOLD,
    ),
    (
        SCALED_1000_SMALL_COLUMNS,
        SCALED_1000_SMALL_THRESHOLD,
    ),
]:

    mask = experiments[columns] > threshold

    experiments[columns] = (
        experiments[columns]
        .mask(
            mask,
            experiments[columns] / 1000,
        )
    )


# ============================================================
# Check selected variables
# ============================================================

missing_variables = [
    column
    for column in SELECTED_VARIABLES
    if column not in experiments.columns
]

if missing_variables:
    raise ValueError(
        f"Selected variables not found: {missing_variables}"
    )


# ============================================================
# Keep relevant columns
# ============================================================

metadata_columns = [
    "Time",
    "elapsed_time",
    "anomaly_label",
    "identifier",
    "phase",
    "batch",
    "operating_point",
    "experiment_type",
    "experiment",
]

experiments = experiments[
    metadata_columns + SELECTED_VARIABLES
]


# ============================================================
# Keep Operation phase
# ============================================================

operation_data = experiments[
    experiments["phase"].str.lower() == "operation"
].copy()


# ============================================================
# Create 60-second windows
# ============================================================

operation_data["window"] = (
    operation_data["elapsed_time"] // WINDOW_SIZE
).astype(int)

operation_data["window_start"] = (
    operation_data["window"] * WINDOW_SIZE
)

operation_data["window_end"] = (
    operation_data["window_start"] + WINDOW_SIZE
)


# ============================================================
# Save operation data
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

operation_path = (
    OUTPUT_DIR / "operation_data.parquet"
)

operation_data.to_parquet(
    operation_path,
    index=False,
)


print("=" * 70)
print("OPERATION DATA")
print("=" * 70)

print(f"Shape:              {operation_data.shape}")
print(
    f"Experiments:        "
    f"{operation_data['identifier'].nunique()}"
)
print(
    f"Variables:          "
    f"{len(SELECTED_VARIABLES)}"
)
print(f"Window size:        {WINDOW_SIZE} seconds")
print(f"Saved to:           {operation_path}")


# ============================================================
# Create raw-signal baseline
# ============================================================

window_counts = (
    operation_data
    .groupby(
        ["identifier", "window"]
    )
    .size()
    .reset_index(name="n_samples")
)

complete_windows = window_counts[
    window_counts["n_samples"] == WINDOW_SIZE
][
    ["identifier", "window"]
]


baseline_data = operation_data.merge(
    complete_windows,
    on=["identifier", "window"],
    how="inner",
)


baseline_data = baseline_data.sort_values(
    [
        "identifier",
        "window",
        "elapsed_time",
    ]
)


# ============================================================
# Flatten complete windows
# ============================================================

baseline_rows = []

for (identifier, window), group in baseline_data.groupby(
    ["identifier", "window"],
    sort=False,
):

    if len(group) != WINDOW_SIZE:
        continue

    row = {
        "identifier": identifier,
        "batch": group["batch"].iloc[0],
        "operating_point": group["operating_point"].iloc[0],
        "experiment": group["experiment"].iloc[0],
        "experiment_type": group["experiment_type"].iloc[0],
        "window": window,
        "window_start": group["window_start"].iloc[0],
        "window_end": group["window_end"].iloc[0],
        "anomaly_label": group["anomaly_label"].iloc[0],
    }

    for variable in SELECTED_VARIABLES:

        values = group[variable].to_numpy()

        for i, value in enumerate(values):

            row[f"{variable}_{i:02d}"] = value

    baseline_rows.append(row)


baseline_raw_windows = pd.DataFrame(
    baseline_rows
)


# ============================================================
# Observable anomaly target
# ============================================================

baseline_raw_windows["observable_anomaly"] = (
    baseline_raw_windows["anomaly_label"]
    .map({
        0: 0,
        1: pd.NA,
        2: 1,
        3: 1,
    })
    .astype("Int64")
)


# ============================================================
# Save baseline
# ============================================================

baseline_path = (
    OUTPUT_DIR / "baseline_raw_windows.parquet"
)

baseline_raw_windows.to_parquet(
    baseline_path,
    index=False,
)


# ============================================================
# Validation
# ============================================================

raw_feature_columns = [
    column
    for column in baseline_raw_windows.columns
    if column.rsplit("_", 1)[-1].isdigit()
]


print("\n" + "=" * 70)
print("RAW-SIGNAL BASELINE")
print("=" * 70)

print(
    f"Shape:                  "
    f"{baseline_raw_windows.shape}"
)

print(
    f"Complete windows:       "
    f"{len(baseline_raw_windows)}"
)

print(
    f"Experiments:            "
    f"{baseline_raw_windows['identifier'].nunique()}"
)

print(
    f"Variables:              "
    f"{len(SELECTED_VARIABLES)}"
)

print(
    f"Raw signal features:    "
    f"{len(raw_feature_columns)}"
)

print(
    f"Expected features:      "
    f"{len(SELECTED_VARIABLES)} × {WINDOW_SIZE} = "
    f"{len(SELECTED_VARIABLES) * WINDOW_SIZE}"
)

print(
    f"Saved to:               "
    f"{baseline_path}"
)

print("\nTarget distribution:")

print(
    baseline_raw_windows["observable_anomaly"]
    .value_counts(dropna=False)
    .sort_index()
)

OPERATION DATA
Shape:              (693790, 42)
Experiments:        119
Variables:          30
Window size:        60 seconds
Saved to:           ..\data\processed\notebooks\operation_data.parquet

RAW-SIGNAL BASELINE
Shape:                  (11512, 1810)
Complete windows:       11512
Experiments:            119
Variables:              30
Raw signal features:    1800
Expected features:      30 × 60 = 1800
Saved to:               ..\data\processed\notebooks\baseline_raw_windows.parquet

Target distribution:
observable_anomaly
0       8887
1       2239
<NA>     386
Name: count, dtype: Int64
